In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [2]:
# Check what sheets are available in the Excel file
xl_file = pd.ExcelFile("../data/SPFmicrodata.xlsx")
print("Available indicators:", xl_file.sheet_names)

# Load only the NGDP sheet
df = pd.read_excel("../data/SPFmicrodata.xlsx", sheet_name="NGDP")
print(f"\nLoaded NGDP sheet: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumn names:", df.columns.tolist())

Available indicators: ['NGDP', 'PGDP', 'CPROF', 'UNEMP', 'EMP', 'INDPROD', 'HOUSING', 'TBILL', 'BOND', 'BAABOND', 'TBOND', 'RGDP', 'RCONSUM', 'RNRESIN', 'RRESINV', 'RFEDGOV', 'RSLGOV', 'RCBI', 'REXPORT', 'CPI5YR', 'PCE5YR', 'CPI10', 'PCE10', 'RGDP10', 'PROD10', 'STOCK10', 'BOND10', 'BILL10', 'PRGDP', 'PRPGDP', 'PRUNEMP', 'PRCCPI', 'PRCPCE', 'RECESS', 'CPI', 'CORECPI', 'PCE', 'COREPCE', 'UBAR', 'SPR_TBOND_TBILL', 'SPR_BAA_AAA', 'SPR_BAA_TBOND', 'SPR_AAA_TBOND', 'CPIF5', 'PCEF5', 'RR1_TBILL_PGDP', 'RR2_TBILL_PGDP', 'RR3_TBILL_PGDP', 'RR1_TBILL_CPI', 'RR2_TBILL_CPI', 'RR3_TBILL_CPI', 'RR1_TBILL_CCPI', 'RR2_TBILL_CCPI', 'RR3_TBILL_CCPI', 'RR1_TBILL_PCE', 'RR2_TBILL_PCE', 'RR3_TBILL_PCE', 'RR1_TBILL_CPCE', 'RR2_TBILL_CPCE', 'RR3_TBILL_CPCE']

Loaded NGDP sheet: 9145 rows × 12 columns

Column names: ['YEAR', 'QUARTER', 'ID', 'INDUSTRY', 'NGDP1', 'NGDP2', 'NGDP3', 'NGDP4', 'NGDP5', 'NGDP6', 'NGDPA', 'NGDPB']


/Users/Parimah/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [3]:
df.head()

,YEAR,QUARTER,ID,INDUSTRY,NGDP1,NGDP2,NGDP3,NGDP4,NGDP5,NGDP6,NGDPA,NGDPB
0,1968,4,1,NaN,871.0,884.0,895.0,907.0,920.0,938.0,NaN,NaN
1,1968,4,2,NaN,871.0,891.0,910.0,929.0,958.0,973.0,NaN,NaN
2,1968,4,3,NaN,871.0,883.0,894.0,906.0,924.0,944.0,NaN,NaN
3,1968,4,4,NaN,871.0,885.0,891.0,902.0,919.0,937.0,NaN,NaN
4,1968,4,5,NaN,871.0,895.0,913.0,935.0,940.0,970.0,NaN,NaN


In [4]:
print(" Number of unique IDs:", df['ID'].nunique())
print(" Number of unique years:", df['YEAR'].nunique())
print(" Number of na values in NGDP6:", df['NGDP6'].isna().sum())

 Number of unique IDs: 462
 Number of unique years: 58
 Number of na values in NGDP6: 974


Forecasts for the quarterly and annual level of nominal GDP. Seasonally
adjusted, annual rate, billions $. Prior to 1992, these are forecasts for
nominal GNP. Annual forecasts are for the annual average of the quarterly
levels.

First survey to include this variable: 1968:Q4


https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/survey-of-professional-forecasters/spf-documentation.pdf?sc_lang=en&hash=8408A4F1BF351A3C268B40F6BC7B95AA


Actual Nominal GDP Data by Quarters 

https://fred.stlouisfed.org/series/GDP

In [5]:
# Check what sheets are available in the Excel file
NGDP_xl_file = pd.ExcelFile("../data/GDP.xlsx")
print("Available indicators:", NGDP_xl_file.sheet_names)

# Load only the NGDP sheet
NGDP_act = pd.read_excel("../data/GDP.xlsx", sheet_name="Quarterly")
# Display the first few rows of the NGDP data
print(NGDP_act.head())

Available indicators: ['README', 'Quarterly']
  observation_date      GDP
0       1947-01-01  243.164
1       1947-04-01  245.968
2       1947-07-01  249.585
3       1947-10-01  259.745
4       1948-01-01  265.742


In [6]:
# Convert the observation_date colum to year and quarter
NGDP_act['YEAR'] = NGDP_act['observation_date'].dt.year
NGDP_act['QUARTER'] = NGDP_act['observation_date'].dt.month.apply(lambda x: (x-1)//3 + 1)
NGDP_act.rename(columns={'NA000334Q': 'NGDP_actual'}, inplace=True)
# only keep the data from 1968 Q3 onwards to match the forecast data
NGDP_act = NGDP_act[(NGDP_act['YEAR'] > 1968) | ((NGDP_act['YEAR'] == 1968) & (NGDP_act['QUARTER'] >= 4))]

NGDP_act.drop(columns=['observation_date'], inplace=True)

NGDP_act.head()

,GDP,YEAR,QUARTER
87,968.030,1968,4
88,993.337,1969,1
89,1009.020,1969,2
90,1029.956,1969,3
91,1038.147,1969,4


In [21]:

# NGDP1 = t-1 (previous quarter historical)
# NGDP2 = t+0 (nowcast, current quarter)
# NGDP3 = t+1, NGDP4 = t+2, NGDP5 = t+3, NGDP6 = t+4
horizon_offsets = {
    'NGDP1': -1,
    'NGDP2':  0,
    'NGDP3':  1,
    'NGDP4':  2,
    'NGDP5':  3,
    'NGDP6':  4,
}

# Build a fast (year, quarter) -> GDP lookup
gdp_lookup = NGDP_act.set_index(['YEAR', 'QUARTER'])['GDP']

errors_df = df[['YEAR', 'QUARTER', 'ID', 'INDUSTRY'] + list(horizon_offsets.keys())].copy()

for col, offset in horizon_offsets.items():
    # Shift survey (YEAR, QUARTER) by offset to get the target quarter
    total    = (errors_df['YEAR'] - 1) * 4 + (errors_df['QUARTER'] - 1) + offset
    t_year   = (total // 4) + 1
    t_qtr    = (total % 4) + 1
    
    gdp_actual = np.array([gdp_lookup.get((y, q), np.nan)
                            for y, q in zip(t_year, t_qtr)])
    
    errors_df[f'error_{col}'] = errors_df[col].values - gdp_actual
    errors_df.drop(columns=[col], inplace=True)

errors_df.head(10)


,YEAR,QUARTER,ID,INDUSTRY,error_NGDP1,error_NGDP2,error_NGDP3,error_NGDP4,error_NGDP5,error_NGDP6
0,1968,4,1,NaN,NaN,-84.03,-98.337,-102.02,-109.956,-100.147
1,1968,4,2,NaN,NaN,-77.03,-83.337,-80.02,-71.956,-65.147
2,1968,4,3,NaN,NaN,-85.03,-99.337,-103.02,-105.956,-94.147
3,1968,4,4,NaN,NaN,-83.03,-102.337,-107.02,-110.956,-101.147
4,1968,4,5,NaN,NaN,-73.03,-80.337,-74.02,-89.956,-68.147
5,1968,4,6,NaN,NaN,-83.03,-98.337,-101.02,-119.956,-113.147
6,1968,4,7,NaN,NaN,-84.03,-98.337,-98.02,-99.956,-89.147
7,1968,4,8,NaN,NaN,-83.03,-105.337,-109.02,-108.956,-102.147
8,1968,4,9,NaN,NaN,-85.03,-100.337,-109.02,-117.956,-108.147
9,1968,4,10,NaN,NaN,-83.03,-98.337,-95.02,-92.956,-81.147


In [ ]:
# let's only keep NGDP3 errors for now
errors_df = errors_df[['YEAR', 'QUARTER', 'ID', 'INDUSTRY', 'error_NGDP3']].copy()
errors_df['error_NGDP3'] = errors_df['error_NGDP3']**2
errors_df.rename(columns={'error_NGDP3': 'Squared Error'}, inplace=True)

# reshape the matrix to have one row per (YEAR, QUARTER) and columns for each ID
errors_df = errors_df.pivot_table(index=['YEAR', 'QUARTER'], columns='ID', values='Squared Error')

errors_df.head()

In [29]:
# now we can calculate the avarage squared error for each forecaster and we can rank the forecasters based on this. 
mean_squared_errors = errors_df.mean()
ranked_forecasters = mean_squared_errors.sort_values()
print("Ranked forecasters based on mean squared error for NGDP3:")
print(ranked_forecasters)


Ranked forecasters based on mean squared error for NGDP3:
ID
150    6.570561e+03
24     1.883376e+04
11     2.106765e+04
29     2.206774e+04
45     2.239510e+04
           ...     
601    5.308944e+06
588    5.373744e+06
591    5.430717e+06
589    5.572543e+06
590    5.997301e+06
Length: 344, dtype: float64


In [33]:
enough

Index([  1,   3,   7,   8,   9,  12,  13,  14,  15,  16,
       ...
       572, 574, 576, 577, 578, 579, 584, 586, 587, 588],
      dtype='int64', name='ID', length=149)

In [34]:
import sys
sys.path.insert(0, '.')
from Bootstrap import rank_confidence_intervals_bootstrap

min_obs = 20
enough  = errors_df.columns[errors_df.notna().sum() >= min_obs]
print(f"Forecasters with >= {min_obs} obs: {len(enough)}")

X_wide = errors_df[enough]
print(f"Before dropna — shape: {X_wide.shape}")

X_wide = X_wide.dropna()
print(f"After dropna  — shape: {X_wide.shape}")


Forecasters with >= 20 obs: 149
Before dropna — shape: (225, 149)
After dropna  — shape: (0, 149)


If After dropna gives 0 rows, the forecasters never all overlap in the same period. The fix is to take only the top N most frequent forecasters.

In [36]:
# Take top N forecasters by number of observations
N = 30  # tune this until After dropna has enough rows
top_ids = errors_df.notna().sum().nlargest(N).index
X_wide  = errors_df[top_ids].dropna()
print(f"Top-{N} forecasters, {X_wide.shape[0]} complete periods")

X = X_wide.values

result = rank_confidence_intervals_bootstrap(X, alpha=0.05, B=2000, seed=42)

summary = pd.DataFrame({
    'ID':         X_wide.columns.tolist(),
    'mean_mse':   result['theta_hat'],
    'rank_lower': result['rank_ci'][:, 0],
    'rank_upper': result['rank_ci'][:, 1],
}).sort_values('mean_mse').reset_index(drop=True)
summary.index += 1
summary


Top-30 forecasters, 0 complete periods


ValueError: Need at least 2 rows.

In [37]:
for N in [3, 5, 8, 10, 15, 20]:
    top_ids  = errors_df.notna().sum().nlargest(N).index
    complete = errors_df[top_ids].dropna().shape[0]
    print(f"N={N:3d}: {complete} complete periods")


N=  3: 50 complete periods
N=  5: 46 complete periods
N=  8: 22 complete periods
N= 10: 4 complete periods
N= 15: 0 complete periods
N= 20: 0 complete periods


In [38]:
import sys
sys.path.insert(0, '.')
from Bootstrap import rank_confidence_intervals_bootstrap

N       = 8
top_ids = errors_df.notna().sum().nlargest(N).index
X_wide  = errors_df[top_ids].dropna()

print(f"Forecasters: {X_wide.shape[1]}, Complete periods: {X_wide.shape[0]}")

X = X_wide.values  # already squared errors, shape (22, 8)

result = rank_confidence_intervals_bootstrap(X, alpha=0.05, B=2000, seed=42)

summary = pd.DataFrame({
    'ID':         X_wide.columns.tolist(),
    'mean_mse':   result['theta_hat'],
    'rank_lower': result['rank_ci'][:, 0],
    'rank_upper': result['rank_ci'][:, 1],
}).sort_values('mean_mse').reset_index(drop=True)
summary.index += 1

print(f"Bootstrap critical value: {result['critical_value']:.3f}\n")
summary


Forecasters: 8, Complete periods: 22
Bootstrap critical value: 4.023



,ID,mean_mse,rank_lower,rank_upper
1,40,728933.840660,4,8
2,84,741169.041142,1,8
3,421,743749.265973,1,8
4,428,746187.818370,1,8
5,433,763604.631755,1,8
6,411,774828.892649,1,7
7,426,788773.387248,1,7
8,65,805302.762697,1,7
